# Which models work well together?

Evaluate every nonempty subset of the five CNNs and XGBoost using the saved seed-42 scores. All 57 ensembles and six individual references use the same 18,714 held-out clips. This exploratory comparison reuses the original test set.

## Load the six independent sets of predictions

The original five-CNN ensemble scores provide a numerical check on the fusion implementation.

In [1]:
from pathlib import Path
import os
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebooks/code").is_dir() and (p / "results/fits.json").is_file())
os.chdir(root)
import csv
import hashlib
import itertools
import json
from pathlib import Path

import numpy as np
from sklearn.metrics import roc_auc_score


NAMES = ("MobileNetV3-Small", "ResNet-18", "EfficientNet-B0", "Compact CNN", "Wide CNN", "XGBoost")
CNN_PARAMS = (1521668, 11172292, 4012096, 177724, 1286964, 0)
MACHINES = ("fan", "pump", "slider", "valve")
evidence = root / "results"
output = root / "runs/ensemble_subsets_seed42"
output.mkdir(exist_ok=True, parents=True)
fits = {r["model"]: r["key"] for r in json.loads((evidence/"fits.json").read_text()) if r["seed"] == 42}
with np.load(evidence/"clip_scores.npz", allow_pickle=False) as archive:
    data = {key: archive[key] for key in archive.files}
a = np.stack([data[fits[n]+"_train"] for n in NAMES])
b = np.stack([data[fits[n]+"_test"] for n in NAMES])
if not (np.isfinite(a).all() and np.isfinite(b).all() and a.min() >= 0 and b.min() >= 0):
    raise ValueError("Invalid input anomaly scores")
a, b = (np.log(np.maximum(values, np.finfo(float).tiny)) for values in (a, b))
y = data["test_label"]
assert not data["train_label"].any() and len(y) == 18714 and y.sum() == 9900

## Calibrate on healthy reference recordings

Standardize log scores separately for each model, machine type and machine ID. The same calibration applies at each noise level.

In [2]:
groups, cells = [], []
for m in MACHINES:
    for mid in np.unique(data["test_id"]):
        tr = np.flatnonzero((data["train_machine"] == m) & (data["train_id"] == mid))
        te = np.flatnonzero((data["test_machine"] == m) & (data["test_id"] == mid))
        groups.append((tr, te))
        for snr in (-6, 0, 6):
            cells.append(te[data["test_snr"][te] == snr])
        mu, sd = a[:, tr].mean(axis=1, keepdims=True), a[:, tr].std(axis=1, keepdims=True)
        assert (sd > 0).all()
        a[:, tr] = (a[:, tr] - mu) / sd
        b[:, te] = (b[:, te] - mu) / sd

## Partition by source recording

A deterministic hash assigns the three noise versions of each source to the same half. We use the first half for exploratory selection and the second half to check the selected configuration.

In [3]:
# Deterministic source-level partition prevents SNR siblings from straddling halves.
sources = [str(m)+"/"+p.split("/", 1)[1] for m,p in zip(data["test_machine"],data["test_path"])]
select = np.array([hashlib.sha256(("subset-study-v1/"+s).encode()).digest()[0] < 128 for s in sources])
all_rows = np.ones(len(y), dtype=bool)
assert len(set(sources))==6238 and not set(np.array(sources)[select]) & set(np.array(sources)[~select])

## Define the metrics

Calculate ROC-AUC and standardized partial AUC within each type, ID and noise group. Calculate recall and false-alarm rate from clip counts at the training-derived threshold.

In [4]:
def summary(y, scores, alarm, cells, keep):
    auc, pauc = [], []
    for cell in cells:
        idx = cell[keep[cell]]
        auc.append(roc_auc_score(y[idx], scores[idx]))
        pauc.append(roc_auc_score(y[idx], scores[idx], max_fpr=0.1))
    tp = int(np.sum(alarm & keep & (y == 1)))
    fp = int(np.sum(alarm & keep & (y == 0)))
    faulty, healthy = int(sum(y[keep])), int(sum(keep) - sum(y[keep]))
    return dict(auc=np.mean(auc), pauc=np.mean(pauc), recall=tp/faulty, fpr=fp/healthy,
                tp=tp, fn=faulty-tp, fp=fp, tn=healthy-fp)

def write_csv(path, rows):
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=rows[0])
        writer.writeheader()
        writer.writerows(rows)

## Evaluate every combination

Average standardized scores with equal weights. Refit the 95th-percentile threshold using only the fused healthy training scores for each type and ID.

In [5]:
rows, selection, check_scores, by_type = [], [], {}, []
for size in range(1, len(NAMES)+1):
    for members in itertools.combinations(range(len(NAMES)), size):
        name = " + ".join(NAMES[i] for i in members)
        header = dict(members=name, size=size, cnn_parameters=sum(CNN_PARAMS[i] for i in members),
                      includes_xgboost=5 in members)
        train, test = a[list(members)].mean(axis=0), b[list(members)].mean(axis=0)
        if members == tuple(range(5)):
            for split, values in [("train", train), ("test", test)]:
                np.testing.assert_allclose(values, data[fits["CNN ensemble"]+"_"+split], atol=1e-11, rtol=1e-11)
        alarm = np.zeros(y.shape, dtype=bool)
        for tr, te in groups:
            alarm[te] = test[te] > np.quantile(train[tr], .95)
        rows.append({**header, **summary(y, test, alarm, cells, all_rows)})
        for i, m in enumerate(MACHINES):
            by_type.append({**header, "machine":m, **summary(y,test,alarm,cells[12*i:12*i+12],data["test_machine"]==m)})
        if size > 1:
            selection.append({**header, **summary(y, test, alarm, cells, select)})
            check_scores[name] = test, alarm

## Check the selected configuration and save results

Choose the first-half leader by partial AUC, breaking ties by ensemble size and name. The output includes all combinations, per-machine results and the grouped check.

In [6]:
best = sorted(selection, key=lambda r: (-r["pauc"],r["size"],r["members"]))[0]
recommended = best["members"]
holdout = {name: summary(y, *check_scores[name], cells, ~select)
           for name in {recommended, " + ".join(NAMES[:5]), " + ".join(NAMES)}}
rows.sort(key=lambda r: -r["pauc"])
write_csv(output/"all_combinations.csv", rows)
write_csv(output/"by_machine.csv", by_type)
study = dict(seed=42, selection_half_clips=int(select.sum()), check_half_clips=int((~select).sum()),
             selected_by_first_half_pauc=best, second_half_results=holdout,
             split="sha256(subset-study-v1/ + machine/source_recording_key), first byte < 128 selects first half",
             caveat="Both halves belong to the project's previously inspected test set.",
             evidence_sha256=hashlib.sha256((evidence/"clip_scores.npz").read_bytes()).hexdigest())
(output/"split_check.json").write_text(json.dumps(study,indent=2)+"\n")

1626

## Inspect the observed leaders

Metrics below are percentages. The companion [results notebook](05_ensemble_findings.ipynb) compares the four-model leader with the original ensemble and a three-model alternative.

In [7]:
for row in rows[:5]:
    print(row["members"])
    print("  " + ", ".join(f"{k} = {100*row[k]:.2f}%" for k in ("auc", "pauc", "recall", "fpr")))

ResNet-18 + EfficientNet-B0 + Wide CNN + XGBoost
  auc = 95.66%, pauc = 90.55%, recall = 80.70%, fpr = 7.42%
ResNet-18 + EfficientNet-B0 + Compact CNN + XGBoost
  auc = 95.58%, pauc = 90.30%, recall = 80.62%, fpr = 7.76%
ResNet-18 + EfficientNet-B0 + Compact CNN + Wide CNN + XGBoost
  auc = 95.40%, pauc = 90.26%, recall = 79.61%, fpr = 7.58%
MobileNetV3-Small + ResNet-18 + EfficientNet-B0 + Wide CNN + XGBoost
  auc = 95.09%, pauc = 90.09%, recall = 80.83%, fpr = 8.53%
ResNet-18 + EfficientNet-B0 + XGBoost
  auc = 95.58%, pauc = 90.08%, recall = 80.66%, fpr = 7.51%
